# Слой PROCESSES: очистка и агрегация

**Вход:** ../raw/building_violations.parquet — raw-слой, одна строка = одно нарушение.

**Выход:** building_violations_processed.parquet — одна строка = одна инспекция объекта (inspection_number).

Целевая метка — inspection_status, результат инспекции. Тяжёлые текстовые столбцы (текст постановления, комментарии инспектора, дубли адреса) для задачи не нужны и не загружаются.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAW = Path('../raw/building_violations.parquet')
OUT = Path('building_violations_processed.parquet')

COLS = ['id', 'violation_last_modified_date', 'violation_date', 'violation_code',
        'violation_status', 'violation_status_date', 'violation_description',
        'inspector_id', 'inspection_number', 'inspection_status', 'inspection_waived',
        'inspection_category', 'department_bureau', 'address', 'property_group',
        'latitude', 'longitude']

missing_cols = [c for c in COLS if c not in set(pq.read_schema(RAW).names)]
assert not missing_cols, f'В raw-слое нет столбцов: {missing_cols}'

df = pd.read_parquet(RAW, columns=COLS)
rows_raw = len(df)
mem_raw = df.memory_usage(deep=True).sum() / 1e6
print(f'строк: {rows_raw:,} | столбцов: {df.shape[1]} | память: {mem_raw:,.0f} МБ')
df.head()

строк: 2,030,101 | столбцов: 17 | память: 480 МБ


,id,violation_last_modified_date,violation_date,violation_code,violation_status,violation_status_date,violation_description,inspector_id,inspection_number,inspection_status,inspection_waived,inspection_category,department_bureau,address,property_group,latitude,longitude
0,7401173,2026-08-31,2025-10-14,CN194039,COMPLIED,2026-08-31,REMOVE,BL00876,14706675,PASSED,N,COMPLAINT,CONSERVATION,6225 N WHIPPLE ST,221530,41.994644,-87.705464
1,7537019,2026-08-28,2026-08-24,PL159017,OPEN,NaT,REPLACE DEFECTIVE PLBG FIXTURE,293822,14927178,FAILED,N,COMPLAINT,CONSERVATION,7300 S WABASH AVE,403784,41.761813,-87.623365
2,7537452,2026-08-31,2026-08-31,CN138106,OPEN,NaT,STOP/REMOVE NUISANCE,478934,14937463,FAILED,N,COMPLAINT,CONSERVATION,10023 S YALE AVE,493332,41.711843,-87.629486
3,1269219,2026-08-31,2006-02-10,CN132016,COMPLIED,2009-07-27,HEAT UNIT ADEQUATELY,BL00943,1399347,FAILED,N,COMPLAINT,CONSERVATION,8031 S DREXEL AVE,20971,41.748543,-87.602531
4,1269220,2026-08-31,2006-02-10,CN046013,COMPLIED,2009-07-27,UNAPPROVED HEATING DEVICE,BL00943,1399347,FAILED,N,COMPLAINT,CONSERVATION,8031 S DREXEL AVE,20971,41.748543,-87.602531


## 1. Дубликаты

Одна и та же запись может выгружаться несколько раз с разной датой изменения. Оставляем последнюю версию по id.

In [2]:
df = df.sort_values('violation_last_modified_date', kind='stable')
dup_full = int(df.duplicated().sum())
dup_id = int(df.duplicated('id', keep='last').sum())
df = df.drop_duplicates('id', keep='last')
print('полных дубликатов строк:', dup_full)
print('повторов id (оставлена последняя версия):', dup_id)

полных дубликатов строк: 0
повторов id (оставлена последняя версия): 0


## 2. Нормализация категорий

Убираем пробелы по краям, приводим к верхнему регистру, пустые строки превращаем в пропуски.

In [3]:
CAT_COLS = ['violation_status', 'inspection_status', 'inspection_waived',
            'inspection_category', 'department_bureau']
ID_COLS = ['violation_code', 'inspector_id', 'inspection_number', 'property_group', 'address']

for c in CAT_COLS:
    df[c] = df[c].astype('string').str.strip().str.upper().replace('', pd.NA)
for c in ID_COLS:
    df[c] = df[c].astype('string').str.strip().replace('', pd.NA)

print(df['inspection_status'].value_counts(dropna=False))
print()
print(df['department_bureau'].value_counts().head(12))

inspection_status
FAILED    1385395
PASSED     389018
CLOSED     255502
HOLD          170
<NA>           16
Name: count, dtype: int64[pyarrow]

department_bureau
CONSERVATION                  1334954
SPECIAL TASK FORCE             148327
ELEVATOR                       136043
DEMOLITION                     119441
ELECTRICAL                      42863
BOILER                          40441
PLUMBING                        39695
NEW CONSTRUCTION                39134
VENTILATION                     36637
REFRIGERATION                   32069
SPECIAL INSPECTION PROGRAM      28996
SIGNS                           15567
Name: count, dtype: int64[pyarrow]


## 3. Даты и координаты

Дата нарушения должна быть между 2006 годом и сегодняшним днём. Дата статуса не может быть раньше даты нарушения. Координаты должны попадать в границы Чикаго.

In [4]:
today = pd.Timestamp.today().normalize()
for c in ['violation_date', 'violation_status_date', 'violation_last_modified_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

bad_date = (df['violation_date'].isna()
            | (df['violation_date'] < pd.Timestamp('2006-01-01'))
            | (df['violation_date'] > today))
df = df[~bad_date].copy()
print('удалено записей без корректной даты нарушения:', int(bad_date.sum()))

bad_status = df['violation_status_date'] < df['violation_date']
df.loc[bad_status, 'violation_status_date'] = pd.NaT
print('дата статуса раньше даты нарушения (обнулена):', int(bad_status.sum()))

in_city = df['latitude'].between(41.6, 42.1) & df['longitude'].between(-88.0, -87.5)
df.loc[~in_city, ['latitude', 'longitude']] = np.nan
print('координаты вне Чикаго или пустые:', int((~in_city).sum()))

удалено записей без корректной даты нарушения: 0
дата статуса раньше даты нарушения (обнулена): 0
координаты вне Чикаго или пустые: 2020


## 4. Пропуски

In [5]:
print((df.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

df['inspection_waived'] = df['inspection_waived'].fillna('N')
for c in ['inspection_category', 'department_bureau', 'violation_status']:
    df[c] = df[c].fillna('UNKNOWN')

before = len(df)
df = df[df['inspection_number'].notna() & df['inspection_status'].notna()].copy()
rows_clean = len(df)
print(f'удалено строк без номера инспекции или статуса: {before - rows_clean:,}')
print(f'строк после очистки: {rows_clean:,}')

violation_status_date           57.48
violation_description            0.73
longitude                        0.10
latitude                         0.10
id                               0.00
violation_status                 0.00
violation_code                   0.00
violation_last_modified_date     0.00
violation_date                   0.00
inspection_number                0.00
dtype: float64


удалено строк без номера инспекции или статуса: 16
строк после очистки: 2,030,085


## 5. Агрегация до уровня инспекции

Одна инспекция содержит несколько нарушений. Сворачиваем их в одну строку: сколько нарушений найдено, сколько осталось открытыми, как быстро их закрывали. Результат инспекции берём из самой свежей записи.

In [6]:
df['is_open'] = df['violation_status'].eq('OPEN').fillna(False).astype('int8')
df['is_complied'] = df['violation_status'].eq('COMPLIED').fillna(False).astype('int8')
df['days_to_status'] = (df['violation_status_date'] - df['violation_date']).dt.days

processed = (
    df.groupby('inspection_number', sort=False)
      .agg(
          inspection_date=('violation_date', 'min'),
          last_modified=('violation_last_modified_date', 'max'),
          property_group=('property_group', 'first'),
          address=('address', 'first'),
          department_bureau=('department_bureau', 'first'),
          inspection_category=('inspection_category', 'first'),
          inspector_id=('inspector_id', 'first'),
          inspection_waived=('inspection_waived', 'first'),
          latitude=('latitude', 'mean'),
          longitude=('longitude', 'mean'),
          n_violations=('id', 'count'),
          n_unique_codes=('violation_code', 'nunique'),
          n_open=('is_open', 'sum'),
          n_complied=('is_complied', 'sum'),
          days_to_status_mean=('days_to_status', 'mean'),
          inspection_status=('inspection_status', 'last'),
      )
      .reset_index()
)
processed['share_open'] = processed['n_open'] / processed['n_violations']

print('Количество строк до агрегации:', f'{len(df):,}')
print('Количество строк после агрегации:', f'{len(processed):,}')
print()
print('Гранулярность: одна строка = одна инспекция объекта (inspection_number)')
print()
print(processed['inspection_status'].value_counts(normalize=True).round(3))

Количество строк до агрегации: 2,030,085
Количество строк после агрегации: 470,057

Гранулярность: одна строка = одна инспекция объекта (inspection_number)

inspection_status
FAILED    0.628
PASSED    0.217
CLOSED    0.155
HOLD        0.0
Name: proportion, dtype: double[pyarrow]


## 6. Оптимизация типов

Категории с небольшим числом значений храним как category, счётчики сжимаем до минимального целого типа, дробные — до float32.

In [7]:
mem_before = processed.memory_usage(deep=True).sum() / 1e6
for c in ['department_bureau', 'inspection_category', 'inspection_waived', 'inspection_status']:
    processed[c] = processed[c].astype('category')
for c in ['n_violations', 'n_unique_codes', 'n_open', 'n_complied']:
    processed[c] = pd.to_numeric(processed[c], downcast='integer')
for c in ['latitude', 'longitude', 'days_to_status_mean', 'share_open']:
    processed[c] = processed[c].astype('float32')
mem_after = processed.memory_usage(deep=True).sum() / 1e6
print(f'память processed: {mem_before:,.1f} МБ → {mem_after:,.1f} МБ')
processed.dtypes

память processed: 97.9 МБ → 52.7 МБ


inspection_number              string
inspection_date        datetime64[us]
last_modified          datetime64[us]
property_group                 string
address                        string
department_bureau            category
inspection_category          category
inspector_id                   string
inspection_waived            category
latitude                      float32
longitude                     float32
n_violations                    int16
n_unique_codes                   int8
n_open                          int16
n_complied                      int16
days_to_status_mean           float32
inspection_status            category
share_open                    float32
dtype: object

## 7. Сохранение

In [8]:
processed.to_parquet(OUT, index=False)
out_mb = OUT.stat().st_size / 1e6
print('файл успешно сохранен:', OUT, f'({out_mb:,.1f} МБ)')

prep_stats = {
    'rows_raw': int(rows_raw),
    'rows_clean': int(rows_clean),
    'rows_processed': int(len(processed)),
    'dup_full': dup_full,
    'dup_id': dup_id,
    'bad_dates': int(bad_date.sum()),
    'bad_status_dates': int(bad_status.sum()),
    'bad_coords': int((~in_city).sum()),
    'mem_raw_mb': round(mem_raw, 1),
    'mem_before_mb': round(mem_before, 1),
    'mem_after_mb': round(mem_after, 1),
    'processed_mb': round(out_mb, 2),
    'status_share': {str(k): round(float(v), 4)
                     for k, v in processed['inspection_status'].value_counts(normalize=True).items()},
}
Path('prep_stats.json').write_text(json.dumps(prep_stats, ensure_ascii=False, indent=2), encoding='utf-8')
processed.head()

файл успешно сохранен: building_violations_processed.parquet (16.0 МБ)


,inspection_number,inspection_date,last_modified,property_group,address,department_bureau,inspection_category,inspector_id,inspection_waived,latitude,longitude,n_violations,n_unique_codes,n_open,n_complied,days_to_status_mean,inspection_status,share_open
0,9647125,2007-11-04,2002-10-31,3021,800 N STATE ST,ELEVATOR,PERMIT,519853,N,41.896717,-87.628349,4,1,4,0,NaN,PASSED,1.0
1,520157,2007-09-18,2003-10-14,2688,55 E ERIE ST,ELEVATOR,PERMIT,519853,N,41.893997,-87.626030,1,1,1,0,NaN,PASSED,1.0
2,535353,2007-12-15,2003-12-23,1821,230 N MICHIGAN AVE,ELEVATOR,PERMIT,541663,N,41.886517,-87.624680,1,1,1,0,NaN,FAILED,1.0
3,1124236,2007-05-01,2005-08-08,547063,1655 S CALIFORNIA BLVD,CONSERVATION,COMPLAINT,BL00340,N,41.857597,-87.695496,1,1,1,0,NaN,FAILED,1.0
4,1128275,2006-01-05,2005-08-15,22634,7250 S LOWE AVE,CONSERVATION,COMPLAINT,BL00876,N,41.761948,-87.640152,12,12,12,0,NaN,FAILED,1.0
